# EcoSort Waste Management Assistant
# Module 8 Summative Lab

## Overview

You are a data scientist at "EcoSort," a technology company that specializes in developing AI solutions for waste management. EcoSort has partnered with Metro City's waste management department to develop an intelligent waste management assistant that can help residents properly dispose of waste items so less time is spent sorting material at facilities.

This assistant needs to:

1. Identify waste materials from images uploaded by residents (CNN)
2. Classify waste items based on text descriptions provided by residents (RNN/Transformer)
3. Generate specific recycling instructions based on identified waste type and city policies (Generative Transformer with RAG)

Your task is to build this integrated system using the RealWaste dataset along with generated text data that simulates real-world waste management operations.

## Part 1: Dataset Exploration and Preparation

**Objectives**
- Examine the image, text, and policy datasets.
- Assess data quality, class balance, and modelling constraints.
- Prepare reproducible pipelines for classification and policy retrieval.

### 1.1 Load and Explore the RealWaste Dataset

This subsection examines the image dataset's structure, class distribution, and visual characteristics before modelling.

#### Code overview: Import all required libraries

Imports and configures the libraries required for data analysis, visualization, modeling, and RAG tasks, ensuring a consistent environment with all necessary dependencies available for the project workflow.


In [ ]:
# Importing all necessary libraries and modules

import os
import json
import random
import pathlib
import warnings
from collections import Counter
from sentence_transformers import SentenceTransformer


import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns


from PIL import Image, ImageFile, UnidentifiedImageError


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input


from IPython.display import display

from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
import faiss


ImageFile.LOAD_TRUNCATED_IMAGES = True


warnings.filterwarnings("ignore")


sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titleweight"] = "bold"



# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print("Libraries imported and random seeds configured successfully.")

#### Code Overview: Load and explore the Realwaste dataset

Loads and explores the RealWaste dataset to examine its structure, category distribution, and image counts, ensuring the data is complete, high quality, and ready for preprocessing, model training, and evaluation..


Creating Image Inventory

Scans the RealWaste dataset to catalog valid images and their categories, creating a structured inventory that supports data exploration, preprocessing, visualization, and model training.


In [ ]:
# Creating image inventory
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Dataset location path
data_dir = (
    Path.cwd().parent
    / "realwaste"
    / "realwaste-main"
    / "RealWaste"
)

# Image formats
valid_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

# Verifying data existence
if not data_dir.exists():
    raise FileNotFoundError(
        f"Dataset directory not found:\n{data_dir.resolve()}"
    )

# Data set location path
data_dir = (
    Path.cwd().parent
    / "realwaste"
    / "realwaste-main"
    / "RealWaste"
)

# Image formats
valid_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

# Verifying data existence
if not data_dir.exists():
    raise FileNotFoundError(
        f"Dataset directory not found:\n{data_dir.resolve()}"
    )


# Getting waste categories


class_names = sorted(
    folder.name
    for folder in data_dir.iterdir()
    if folder.is_dir()
)

print(f"Dataset Location:\n{data_dir.resolve()}")
print(f"\nNumber of Categories: {len(class_names)}")
print("\nCategories:")
print(class_names)

if len(class_names) != 9:
    print(
        f"\nWARNING: Expected 9 categories but found {len(class_names)}"
    )


# Counting Images


image_counts = {}

total_images = 0

for category in class_names:

    category_path = data_dir / category

    count = len(
        [
            f
            for f in category_path.iterdir()
            if f.suffix.lower() in valid_extensions
        ]
    )

    image_counts[category] = count
    total_images += count


# Summary table 

summary_df = pd.DataFrame(
    {
        "Category": image_counts.keys(),
        "Image Count": image_counts.values()
    }
)

display(summary_df.sort_values("Category"))

print("\nTotal Images:", total_images)

# Plot Class Distribution

plt.figure(figsize=(10, 5))

sns.barplot(
    data=summary_df,
    x="Category",
    y="Image Count"
)

plt.title("RealWaste Class Distribution")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


#### Code Overview: Visualizing the category pattern and distribution 
Summarizes and visualizes image counts across waste categories to assess class balance and identify categories that may require additional modeling attention.


In [ ]:

## Visualizing category distributions 

class_distribution = pd.DataFrame(
    {
        "category": list(image_counts.keys()),
        "image_count": list(image_counts.values())
    }
)

display(class_distribution)


# Visualize Category Distribution



plt.figure(figsize=(12, 6))

bar_plot = sns.barplot(
    data=class_distribution,
    x="category",
    y="image_count",
    color="#0B6E4F"
)

# Adding labels above bars
for container in bar_plot.containers:
    bar_plot.bar_label(
        container,
        fmt="%.0f",
        padding=3
    )

plt.title("Distribution of Images Across Waste Categories")
plt.xlabel("Waste Category")
plt.ylabel("Number of Images")
plt.xticks(rotation=35, ha="right")

plt.tight_layout()
plt.show()


# Calculating Class Imbalance


largest_class = class_distribution.loc[
    class_distribution["image_count"].idxmax()
]

smallest_class = class_distribution.loc[
    class_distribution["image_count"].idxmin()
]

imbalance_ratio = (
    largest_class["image_count"]
    / smallest_class["image_count"]
)

print(
    f"\nLargest category: "
    f"{largest_class['category']} "
    f"({largest_class['image_count']:,} images)"
)

print(
    f"Smallest category: "
    f"{smallest_class['category']} "
    f"({smallest_class['image_count']:,} images)"
)

print(
    f"Largest-to-smallest class ratio: "
    f"{imbalance_ratio:.2f}"
)

#### Code Overview: Display one representative sample image from each category
Displays one representative image from each waste category to support visual comparison of category characteristics and potential similarities.


In [ ]:
## Displaying representative images from each category
from PIL import Image

SEED = 42

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.flatten()

for idx, category in enumerate(sorted(image_df["category"].unique())):

    try:
        sample_path = (
            image_df.loc[
                image_df["category"] == category,
                "image_path"
            ]
            .sample(n=1, random_state=SEED)
            .iloc[0]
        )

        img = Image.open(sample_path).convert("RGB")

        axes[idx].imshow(img)
        axes[idx].set_title(category, fontsize=12)
        axes[idx].axis("off")

    except Exception as e:
        axes[idx].text(
            0.5, 0.5,
            f"Error\n{category}",
            ha="center",
            va="center"
        )
        axes[idx].axis("off")
        print(f"Error loading image for {category}: {e}")

plt.suptitle("One Sample Image from Each Waste Category", fontsize=16)
plt.tight_layout()
plt.show()

#### Code Overview: Checking image characteristics and file quality
Examines image properties and validates file integrity to identify invalid files and ensure consistent image preprocessing.

In [ ]:

## Analyzing image characteristics and file quality

# Store image dimensions and file-validation results
image_metadata = []
invalid_images = []

for row in image_df.itertuples(index=False):
    try:
        # Open the image and capture its characteristics
        with Image.open(row.image_path) as image:
            width, height = image.size

            image_metadata.append(
                {
                    "image_path": row.image_path,
                    "category": row.category,
                    "width": width,
                    "height": height,
                    "aspect_ratio": width / height if height else np.nan,
                    "color_mode": image.mode,
                    "file_size_kb": os.path.getsize(row.image_path) / 1024
                }
            )

            # Verify that the file is not structurally corrupted
            image.verify()

    except (
        UnidentifiedImageError,
        OSError,
        ValueError,
        ZeroDivisionError
    ) as error:
        invalid_images.append(
            {
                "image_path": row.image_path,
                "category": row.category,
                "error": str(error)
            }
        )

# Convert valid image metadata into a DataFrame
image_metadata_df = pd.DataFrame(image_metadata)

# Convert invalid-image results into a DataFrame
invalid_images_df = pd.DataFrame(invalid_images)

print(f"Valid images inspected: {len(image_metadata_df):,}")
print(f"Unreadable or corrupted images: {len(invalid_images_df):,}")

# Display invalid files only if any were detected
if not invalid_images_df.empty:
    display(invalid_images_df.head())

#### Code Overview: Summarize image, dimensions and file sizes
Analyzes image dimensions, aspect ratios, file sizes, and color modes to assess dataset consistency and guide preprocessing decisions such as resizing and color standardization.


In [ ]:
# Summarizing image characteristics

image_summary = (
    image_metadata_df[
        ["width", "height", "aspect_ratio", "file_size_kb"]
    ]
    .describe()
    .round(2)
)

display(image_summary)

print("\nMost common image resolutions:")

resolution_counts = (
    image_metadata_df
    .assign(
        resolution=image_metadata_df["width"].astype(str)
        + " × "
        + image_metadata_df["height"].astype(str)
    )["resolution"]
    .value_counts()
    .head(10)
    .rename_axis("resolution")
    .reset_index(name="image_count")
)

display(resolution_counts)

print("\nImage color modes:")

color_mode_counts = (
    image_metadata_df["color_mode"]
    .value_counts()
    .rename_axis("color_mode")
    .reset_index(name="image_count")
)

display(color_mode_counts)

#### Code Overview: Visualizing image width, height and aspect ratio

Visualizes the distributions of image dimensions and aspect ratios to assess variability and inform resizing and preprocessing requirements before CNN training.

In [ ]:
## Visualizing image dimensions and aspect ratios

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Width distribution
sns.histplot(
    data=image_metadata_df,
    x="width",
    bins=30,
    color="#0B6E4F",
    ax=axes[0]
)
axes[0].set_title("Image Width Distribution")
axes[0].set_xlabel("Width in Pixels")
axes[0].set_ylabel("Number of Images")

# Height distribution
sns.histplot(
    data=image_metadata_df,
    x="height",
    bins=30,
    color="#D99201",
    ax=axes[1]
)
axes[1].set_title("Image Height Distribution")
axes[1].set_xlabel("Height in Pixels")
axes[1].set_ylabel("Number of Images")

# Aspect-ratio distribution
sns.histplot(
    data=image_metadata_df,
    x="aspect_ratio",
    bins=30,
    color="#006D77",
    ax=axes[2]
)
axes[2].axvline(
    1.0,
    color="red",
    linestyle="--",
    label="Square image"
)
axes[2].set_title("Image Aspect-Ratio Distribution")
axes[2].set_xlabel("Width-to-Height Ratio")
axes[2].set_ylabel("Number of Images")
axes[2].legend()

plt.tight_layout()
plt.show()

#### Code Overview: Compare image characteristics by waste category 
Compares image counts, dimensions, aspect ratios, and file sizes across waste categories to identify differences that may affect preprocessing and model performance

In [ ]:
## Comparing image characteristics by waste category

category_characteristics = (
    image_metadata_df
    .groupby("category")
    .agg(
        image_count=("image_path", "count"),
        median_width=("width", "median"),
        median_height=("height", "median"),
        median_aspect_ratio=("aspect_ratio", "median"),
        median_file_size_kb=("file_size_kb", "median")
    )
    .reindex(class_names)
    .round(2)
    .reset_index()
)

display(category_characteristics)

### Comments on data characteristics and modelling challenges

The image dataset exhibits moderate class imbalance, with Plastic containing 921 images and Textile Trash only 318 images.- Visually similar categories such as Glass and Miscellaneous Trash may increase classification difficulty.- Real-world waste items contain varying lighting conditions, backgrounds, and object orientations.- Some categories contain heterogeneous objects, increasing intra-class variation.
These observations justify the use of transfer learning, image augmentation, class-level evaluation metrics, and regularization techniques throughout model development.

The image dataset comprises 4,752 valid RGB images across nine classes with consistent dimensions (524 × 524 pixels), though moderate class imbalance and real-world visual variation support the use of augmentation, transfer learning, and class-level evaluation.


### 1.2 Explore Text Datasets

This subsection assesses the waste descriptions and policy documents for completeness, coverage, and modelling suitability.

#### Code Overview: Loading and exploring the waste description dataset 

Loads and examines the waste description text dataset to verify its structure, content, and quality before text exploration and classification.


In [ ]:

# Loading and exploring the waste description dataset

from pathlib import Path
import pandas as pd

# Locate file
csv_file = Path.cwd().parent / "waste_descriptions.csv"

print("Dataset Path:")
print(csv_file)

# Verify file exists
assert csv_file.exists(), f"File not found: {csv_file}"

# Load dataset
waste_df = pd.read_csv(csv_file)


# Dataset Overview


print("=" * 60)
print("WASTE DESCRIPTION DATASET")
print("=" * 60)

print(f"\nDataset Shape: {waste_df.shape}")

print("\nColumns:")
for col in waste_df.columns:
    print(f"- {col}")

print("\nMissing Values:")
display(waste_df.isnull().sum())

print("\nSample Records:")
display(waste_df.head())

# =====================================================
# Additional Exploration
# =====================================================

print("\nData Types:")
display(waste_df.dtypes)

# Category distribution
if "category" in waste_df.columns:

    print("\nCategory Distribution:")

    display(
        waste_df["category"]
        .value_counts()
        .to_frame("count")
    )

# Description lengths
if "description" in waste_df.columns:

    waste_df["word_count"] = (
        waste_df["description"]
        .astype(str)
        .str.split()
        .str.len()
    )

    print("\nDescription Length Statistics:")

    display(
        waste_df["word_count"]
        .describe()
    )

#### Code Overview: Text dataset category distribution 

Summarizes and visualizes category counts to assess class balance and identify categories that may require additional modeling attention.


In [ ]:
## Category Distribution Visualization

category_counts = (
    waste_df["category"]
    .value_counts()
    .sort_values(ascending=False)
)

display(category_counts)

plt.figure(figsize=(10,6))

ax = sns.barplot(
    x=category_counts.index,
    y=category_counts.values,
    color="#0B6E4F"
)

for container in ax.containers:
    ax.bar_label(container)

plt.title("Distribution of Waste Categories")
plt.xlabel("Category")
plt.ylabel("Number of Records")
plt.xticks(rotation=35)

plt.tight_layout()
plt.show()

#### Code Overview: Analysis of text length
Analyzes text length and vocabulary patterns to assess data quality and guide NLP feature engineering and model development.

In [ ]:
## Analysing text length in descriptions

# Calculate word counts
waste_df["description_word_count"] = (
    waste_df["description"]
    .astype(str)
    .str.split()
    .str.len()
)

display(
    waste_df["description_word_count"]
    .describe()
    .round(2)
)

plt.figure(figsize=(10,6))

sns.histplot(
    waste_df["description_word_count"],
    bins=30,
    color="#006D77"
)

plt.title("Distribution of Description Lengths")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.show()

#### Code Overview: Analysis of word frequencies 

Analyzes word frequencies in waste descriptions to identify dominant vocabulary patterns and assess their usefulness for distinguishing waste categories.

In [ ]:
## Analyzing common words in descriptions

from collections import Counter
import re

all_text = " ".join(
    waste_df["description"]
    .astype(str)
    .str.lower()
)

tokens = re.findall(r"\b[a-z]+\b", all_text)

word_freq = Counter(tokens)

common_words = pd.DataFrame(
    word_freq.most_common(20),
    columns=["Word", "Frequency"]
)

display(common_words)

plt.figure(figsize=(10,6))

sns.barplot(
    data=common_words,
    x="Frequency",
    y="Word",
    color="#D99201"
)

plt.title("Top 20 Most Common Words in Waste Descriptions")

plt.show()

#### Code Overview: Common Confusion analysis

Analyzes common category confusions to identify potentially ambiguous waste types and highlight text classification challenges.


In [ ]:
## Checking records containing common confusion information

print(
    "Records containing common confusion information:"
)

print(
    waste_df["common_confusion"]
    .notna()
    .sum()
)

display(
    waste_df[
        [
            "category",
            "common_confusion"
        ]
    ]
    .dropna()
    .head(10)
)

#### Code Brief: Loading and exploring policy documents  

Loads and structures waste policy documents to prepare authoritative content for analysis and retrieval.


In [ ]:
## Loading and exploring the waste policy documents dataset
from pathlib import Path
import json

json_file = Path.cwd().parent / "waste_policy_documents.json"

print("Policy File:")
print(json_file)

with open(
    json_file,
    "r",
    encoding="utf-8"
) as f:
    policy_docs = json.load(f)

print("=" * 60)
print("POLICY DOCUMENTS")
print("=" * 60)

print(f"Number of Documents: {len(policy_docs)}")

print("\nFirst Document:")
display(pd.DataFrame([policy_docs[0]]))

#### Code Overview: Analysis of policy coverage 

Analyzes policy coverage across waste categories to assess information availability and identify gaps that may affect retrieval and content generation.

In [ ]:
## Analysing policy coverage by waste category


policy_summary = pd.DataFrame(policy_docs)

display(
    policy_summary[
        [
            "policy_id",
            "policy_type",
            "categories_covered",
            "effective_date"
        ]
    ]
)

# Count policy coverage by waste category
policy_categories = []

for doc in policy_docs:
    policy_categories.extend(
        doc["categories_covered"]
    )

coverage_df = pd.Series(
    policy_categories
).value_counts()

display(coverage_df)

plt.figure(figsize=(10,6))

sns.barplot(
    x=coverage_df.index,
    y=coverage_df.values,
    color="#006D77"
)

plt.title("Policy Coverage by Waste Category")
plt.xlabel("Category")
plt.ylabel("Number of Policies")

plt.xticks(rotation=35)

plt.tight_layout()
plt.show()

#### Code Overview: Policy document text analysis 

Analyzes policy document lengths to assess the volume and consistency of information available for semantic retrieval and content generation.


In [ ]:
## Analysing policy document text 

policy_summary["document_length"] = (
    policy_summary["document_text"]
    .astype(str)
    .str.split()
    .str.len()
)

display(
    policy_summary[
        "document_length"
    ]
    .describe()
    .round(2)
)

plt.figure(figsize=(10,6))

sns.histplot(
    policy_summary["document_length"],
    bins=15,
    color="#0B6E4F"
)

plt.title("Policy Document Length Distribution")
plt.xlabel("Word Count")
plt.ylabel("Number of Documents")

plt.show()

#### Comments on text dataset

The text dataset contains 5,000 short descriptions with a balanced class distribution, though the common_confusion field is incomplete and best suited for qualitative analysis. The 14 policy documents cover all waste categories, providing broad but uneven support for retrieval and content generation.

### 1.3 Create Data Pipelines

This subsection converts the three datasets into model-ready inputs for the CNN, text classifier, and retrieval system.

Code Overview: Create training, validation, and test image sets

Loads, resizes, and encodes labeled images to create reproducible training, validation, and test datasets for CNN development and evaluation.

In [ ]:
# Run this code to setup the images properly into train, validation, and test sets

import pathlib
from pathlib import Path
data_dir = Path("../realwaste/realwaste-main/RealWaste")
# Parameters
BATCH_SIZE = 32
IMG_HEIGHT = 224
IMG_WIDTH = 224

# Calculate the total number of classes automatically from the directory structure
num_classes = len([item for item in data_dir.glob('*') if item.is_dir()])
print(f"Number of classes: {num_classes}")

# List all class folders
class_names = sorted([item.name for item in data_dir.glob('*') if item.is_dir()])
print(f"Class names: {class_names}")

# Count all images
image_count = len(list(data_dir.glob('*/*.jpg'))) + len(list(data_dir.glob('*/*.png')))
print(f"Total images found: {image_count}")

# Create a dataset using tf.keras.utils.image_dataset_from_directory
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,  # 20% for validation
    subset="training",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',  
    shuffle=True
)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,  # 20% for validation
    subset="validation",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',  
    shuffle=True
)

# Create a separate test dataset by taking part of the validation set

val_batches = tf.data.experimental.cardinality(validation_ds)
test_dataset = validation_ds.take(val_batches // 2)
validation_ds = validation_ds.skip(val_batches // 2)

print(f"Number of training batches: {tf.data.experimental.cardinality(train_ds)}")
print(f"Number of validation batches: {tf.data.experimental.cardinality(validation_ds)}")
print(f"Number of test batches: {tf.data.experimental.cardinality(test_dataset)}")

# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

#### Code Overview: Create the text preprocessing pipeline

Cleans and tokenizes waste descriptions, encodes labels, and creates stratified training, validation, and test datasets to produce consistent inputs for text classification while preserving category balance across data splits.


In [ ]:
## Creating the text preprocessing pipeline

# Creating a modelling copy of the waste description dataset
text_df = waste_df.copy()

# Retain records with a valid description and category
text_df = text_df.dropna(
    subset=["description", "category"]
).copy()

# Convert descriptions to lowercase and normalize spacing
text_df["clean_description"] = (
    text_df["description"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)


text_df = text_df[
    text_df["clean_description"].str.len() > 0
].reset_index(drop=True)

# Confirming that text categories match the image categories
text_class_names = sorted(
    text_df["category"].unique()
)

print(f"Text categories: {text_class_names}")
print(f"Image categories: {class_names}")

assert text_class_names == class_names, (
    "The text and image categories do not match."
)

# Creating a consistent category-to-number mapping
text_label_encoder = LabelEncoder()

text_df["label"] = text_label_encoder.fit_transform(
    text_df["category"]
)

# Creating an 80% training set and a 20% holdout set
text_train_df, text_holdout_df = train_test_split(
    text_df,
    test_size=0.20,
    stratify=text_df["category"],
    random_state=SEED
)

# Dividing the holdout data equally into validation and test sets
text_validation_df, text_test_df = train_test_split(
    text_holdout_df,
    test_size=0.50,
    stratify=text_holdout_df["category"],
    random_state=SEED
)

# Resetting indexes after splitting
text_train_df = text_train_df.reset_index(drop=True)
text_validation_df = text_validation_df.reset_index(drop=True)
text_test_df = text_test_df.reset_index(drop=True)

# Summarizing the resulting text splits
text_split_summary = pd.DataFrame(
    {
        "Split": [
            "Training",
            "Validation",
            "Test"
        ],
        "Records": [
            len(text_train_df),
            len(text_validation_df),
            len(text_test_df)
        ]
    }
)

text_split_summary["Percentage"] = (
    text_split_summary["Records"]
    / len(text_df)
    * 100
).round(2)

display(text_split_summary)

# Displaying category distribution across the splits
text_split_distribution = pd.concat(
    [
        text_train_df["category"]
        .value_counts()
        .rename("Training"),

        text_validation_df["category"]
        .value_counts()
        .rename("Validation"),

        text_test_df["category"]
        .value_counts()
        .rename("Test")
    ],
    axis=1
).fillna(0).astype(int).reindex(class_names)

display(text_split_distribution)

# Load the DistilBERT tokenizer
from transformers import AutoTokenizer

text_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

# Using a practical sequence length for the short descriptions
MAX_TEXT_LENGTH = 64


def tokenize_descriptions(dataframe):
    """
    Tokenizes cleaned waste descriptions for DistilBERT.
    """

    return text_tokenizer(
        dataframe["clean_description"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
        return_tensors="np"
    )

# Tokenizing each text split separately
train_text_encoding = tokenize_descriptions(
    text_train_df
)

validation_text_encoding = tokenize_descriptions(
    text_validation_df
)

test_text_encoding = tokenize_descriptions(
    text_test_df
)

print("Text preprocessing completed successfully.")
print(
    "Training token shape:",
    train_text_encoding["input_ids"].shape
)
print(
    "Validation token shape:",
    validation_text_encoding["input_ids"].shape
)
print(
    "Test token shape:",
    test_text_encoding["input_ids"].shape
)

### Reflection on model selection 

Although DistilBERT tokenization was explored, the short waste descriptions (≈5 words on average) were more effectively and efficiently classified using TF-IDF with Logistic Regression.

#### Code Overview: Prepare policy documents for RAG 
Validates, embeds, and indexes policy documents in a FAISS vector database to create a searchable knowledge base for retrieving relevant waste management guidance.

In [ ]:
## Preparing policy documents for retrieval-augmented generation (RAG)
import faiss
from sentence_transformers import SentenceTransformer
# Converting the loaded policy documents into a DataFrame
policy_df = pd.DataFrame(policy_docs).copy()

# Validating fields required by the retrieval pipeline
required_policy_columns = {
    "policy_id",
    "policy_type",
    "categories_covered",
    "document_text",
    "jurisdiction"
}

missing_policy_columns = (
    required_policy_columns
    - set(policy_df.columns)
)

if missing_policy_columns:
    raise ValueError(
        "Missing required policy columns: "
        f"{sorted(missing_policy_columns)}"
    )

# Replacing missing text values
policy_df["policy_type"] = (
    policy_df["policy_type"]
    .fillna("")
    .astype(str)
)

policy_df["document_text"] = (
    policy_df["document_text"]
    .fillna("")
    .astype(str)
)

policy_df["jurisdiction"] = (
    policy_df["jurisdiction"]
    .fillna("")
    .astype(str)
)

# Converting category lists into readable text
policy_df["category_text"] = (
    policy_df["categories_covered"]
    .apply(
        lambda categories: ", ".join(categories)
        if isinstance(categories, list)
        else str(categories)
    )
)

# Combining the relevant fields into retrieval-ready documents
policy_df["retrieval_text"] = (
    "Policy: "
    + policy_df["policy_type"]
    + "\nCategories: "
    + policy_df["category_text"]
    + "\nJurisdiction: "
    + policy_df["jurisdiction"]
    + "\nGuidance: "
    + policy_df["document_text"]
)

# Removing empty and duplicate policy documents
policy_df = (
    policy_df[
        policy_df["retrieval_text"]
        .str.strip()
        .str.len() > 0
    ]
    .drop_duplicates(
        subset=["retrieval_text"]
    )
    .reset_index(drop=True)
)

print(
    "Policy documents prepared:",
    len(policy_df)
)

display(
    policy_df[
        [
            "policy_id",
            "policy_type",
            "category_text",
            "effective_date",
            "jurisdiction"
        ]
    ].head()
)

# Loading the sentence embedding model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# Generating normalized semantic embeddings
policy_embeddings = embedding_model.encode(
    policy_df["retrieval_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Converting embeddings to the data type required by FAISS
policy_embeddings = np.asarray(
    policy_embeddings,
    dtype="float32"
)

# Creating a cosine-similarity retrieval index
embedding_dimension = policy_embeddings.shape[1]

policy_index = faiss.IndexFlatIP(
    embedding_dimension
)

# Adding all policy embeddings to the index
policy_index.add(policy_embeddings)

print("\nPolicy retrieval index created successfully.")
print(f"Indexed documents: {policy_index.ntotal}")
print(f"Embedding dimensions: {embedding_dimension}")

#### Code Brief: Validate the policy retrieval pipeline 
Tests the semantic retrieval pipeline by ranking and returning the most relevant policy documents for a query, ensuring accurate policy retrieval for waste management guidance.


In [ ]:
# Validating the policy retrieval pipeline with a sample query


def retrieve_policy_documents(query, top_k=3):
    """
    Retrieves the policy documents most relevant to a query.

    Args:
        query: Waste-management question or search text.
        top_k: Number of policy documents to retrieve.

    Returns:
        DataFrame containing the highest-ranked policies.
    """

    # Limiting retrieval to the number of available documents
    top_k = min(top_k, len(policy_df))

    # Converting the query into a normalized embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    # Searching for the most similar policy documents
    similarity_scores, document_indices = (
        policy_index.search(
            query_embedding,
            top_k
        )
    )

    # Building a readable results table
    retrieved_documents = policy_df.iloc[
        document_indices[0]
    ][
        [
            "policy_id",
            "policy_type",
            "category_text",
            "document_text"
        ]
    ].copy()

    retrieved_documents.insert(
        3,
        "similarity_score",
        similarity_scores[0].round(4)
    )

    return retrieved_documents.reset_index(
        drop=True
    )


# Testing retrieval using a sample recycling query
sample_query = "How should a glass bottle be recycled?"

sample_retrieval = retrieve_policy_documents(
    sample_query,
    top_k=3
)

print(f"Sample query: {sample_query}")
display(sample_retrieval)

#### Data preparation and RAG readiness assessment 
The image pipeline successfully generated 119 training, 15 validation, and 15 test batches at 224 × 224 resolution, while the text dataset was stratified into 4,000 training, 500 validation, and 500 test records. Additionally, 14 policy documents were embedded into a FAISS index, demonstrating effective semantic retrieval by correctly ranking the Glass Recycling Guidelines as the most relevant result for a sample glass-related query.

# Simple RAG Response Generator 

Tuning the dataset

In [ ]:
# Training data for generative model

generator_examples = []

for _, row in policy_df.iterrows():

    categories = (
        row["category_text"]
        .split(", ")
    )

    for category in categories:

        generator_examples.append({
            "input_text":
                f"How should {category} be recycled?",

            "target_text":
                row["document_text"]
        })

generator_df = pd.DataFrame(
    generator_examples
)

display(generator_df.head())

Loading FLAN-T5

In [35]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

Loading the Generator Model

In [36]:
GENERATOR_MODEL_NAME = (
    "google/flan-t5-small"
)

generator_tokenizer = (
    AutoTokenizer.from_pretrained(
        GENERATOR_MODEL_NAME
    )
)

generator_model = (
    AutoModelForSeq2SeqLM.from_pretrained(
        GENERATOR_MODEL_NAME
    )
)

Tokenizing the generator dataset

In [ ]:
generator_dataset = Dataset.from_pandas(
    generator_df
)

MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 256

def preprocess_generator(example):

    model_inputs = (
        generator_tokenizer(
            example["input_text"],
            truncation=True,
            max_length=MAX_INPUT_LENGTH
        )
    )

    labels = (
        generator_tokenizer(
            example["target_text"],
            truncation=True,
            max_length=MAX_TARGET_LENGTH
        )
    )

    model_inputs["labels"] = (
        labels["input_ids"]
    )

    return model_inputs

generator_dataset = (
    generator_dataset.map(
        preprocess_generator
    )
)

Fine tuning the generator model 

In [38]:
def preprocess_generator_examples(examples):
    """
    Tokenize generator inputs and target responses.
    
    Replace 'input_text' and 'target_text' below if your
    dataset uses different column names.
    """

    model_inputs = generator_tokenizer(
        examples["input_text"],
        max_length=256,
        truncation=True
    )

    labels = generator_tokenizer(
        text_target=examples["target_text"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
generator_dataset = generator_dataset.map(
    preprocess_generator_examples,
    batched=True,
    remove_columns=generator_dataset.column_names
)

In [40]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

MODEL_NAME = "google/flan-t5-base"

generator_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

print("Generator tokenizer loaded")
print("Generator model loaded")

Generator tokenizer loaded
Generator model loaded


In [ ]:
from transformers import DataCollatorForSeq2Seq

generator_data_collator = DataCollatorForSeq2Seq(
    tokenizer=generator_tokenizer,
    model=generator_model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

generator_training_args = TrainingArguments(
    output_dir="./flan_generator",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",
    remove_unused_columns=True
)

generator_trainer = Trainer(
    model=generator_model,
    args=generator_training_args,
    train_dataset=generator_dataset,
    data_collator=generator_data_collator
)

generator_trainer.train()

Saving the model  

In [42]:
generator_model.save_pretrained("./flan_generator_final")
generator_tokenizer.save_pretrained("./flan_generator_final")

('./flan_generator_final\\tokenizer_config.json',
 './flan_generator_final\\special_tokens_map.json',
 './flan_generator_final\\spiece.model',
 './flan_generator_final\\added_tokens.json',
 './flan_generator_final\\tokenizer.json')